# Content-based recommendations
Student information here!!!

In [13]:
from IPython.core.display import HTML
from movie_display import movie_display
import pandas as pd
import numpy as np
import nltk

# Example of showing movie information

In [14]:
# Load movies into a dataframe
df = pd.read_json('../data/movielens/imdbdata.json', orient='columns')

# Display information about some movies
HTML(movie_display.show([df.iloc[0], df.iloc[1], df.iloc[2], df.iloc[3], df.iloc[4]]))

The goal is to recommend based on the content of the movies, hence as the assigment suggests it is best to use features like plot, title, writer, genre which are explaining the content of the movie. For each of these features, different filters are applied, as given below:  
- plot: free field, punctuation needs to be removed, make all words lowercase, stem words, apply all filters 
- title: text field
- writer: word field
- genre:

The methods clean_text and clean_list are created so that they can be reused during the filtering of the features. The clean text method finds all words that contain a digit and remve any punctuation, then it transforms the word to lowercase so it is easier for the vectorizer. 

In this case we also stem the words if they are not a stopword, otherwise we remove them.  

The clean list method deals with lists, and it cleans them up. It removed paranthesis if there were any and in cases where there is an actor name, it replaces the empty space with _. 

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

movies_df = pd.read_json("../data/movielens/imdbdata.json").replace("N/A", "")
sw = set(stopwords.words("english"))
stemmer = SnowballStemmer("english")

# remove punctuation, lowercase, remove stopwords, and stem words
def clean_text(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return " ".join(stemmer.stem(w) for w in words if w not in sw)

# removes unecessary information like the parantheses and converts to lowercase with underscores instead of spaces
def clean_list(text):
    text = re.sub(r"\(.*?\)", "", text) 
    names = [n.lower().strip().replace(" ", "_") for n in text.split(",")]
    return list(dict.fromkeys(n for n in names if n))

We now choose the fields we want to use in the recommendation system. They get stored along with the vectorizer and its settings, the documents with the associated column and the weight of the field inside a dictionary.

Then we create a variable of the type array where we would store the similarity between the features. This variable acts as the matrix for the relation between all the fields, it has shape of NxN - the number of the movies document. This array is later populated with the sum of all weights of the features times the cosine similarity between all the features.

Additionally for the other merging strategy I used the tfidf strategy. The difference with this one is that we calculate the term frequency in the document and divide it with the whole number of terms in the document, after which we inverse it. The default function cosine_similarity this for us. 

In [ ]:
keep = lambda tokens: tokens

# create a vector, store the docs, and the weight for each field
fields = {
    "plot":   (TfidfVectorizer(ngram_range=(1, 2), min_df=2, dtype=np.float32),
               movies_df["Plot"].map(clean_text),    0.40),
    "title":  (TfidfVectorizer(dtype=np.float32),
               movies_df["Title"],   0.20),
    "writer": (TfidfVectorizer(analyzer=keep, dtype=np.float32),
               movies_df["Writer"].map(clean_list),  0.20),
    "genre":  (TfidfVectorizer(analyzer=keep, dtype=np.float32),
               movies_df["Genre"].map(clean_list),   0.20),
}

tfidif_matrices = {}
# Weighted sum of the per-field cosine similarities (added one at a time to save memory)
n = len(movies_df)
merged_sim = np.zeros((n, n), dtype=np.float32)

for name, (vec, docs, weight) in fields.items():
    tfidf = vec.fit_transform(docs)
    tfidif_matrices[name] = tfidf
    cosine_sim = cosine_similarity(tfidf)
    merged_sim += weight * cosine_sim

total = sum(w for _, _, w in fields.values())
weights = {name: w/total for name,(_,_,w) in fields.items()}
features = hstack([np.sqrt(weights[name])*tfidif_matrices[name] for name in fields]).tocsr()

Finally we get to use the merged_sim variable, by passing the index of the movie in as the column name. We sort the result in descending order and show the top best n() recommendations.

In [16]:
def recommend(index, n_recs=10):
    scores = merged_sim[index].copy()
    scores[index] = -1                       # never recommend the movie itself
    top = np.argsort(-scores)[:n_recs]
    return [movies_df.iloc[i] for i in top]

# small example to show it works
HTML(movie_display.show([movies_df.iloc[1]] + recommend(1, 5)))   # Toy Story + 3 recs

...


In [17]:
def recommend_from_watched(watched, n_recs=5):
    profile = np.asarray(features[watched].mean(axis=0))      # average vector = user profile
    scores = cosine_similarity(profile, features).ravel()     # compare the profile with all movies
    scores[watched] = -1                                      # never recommend watched movies
    top = np.argsort(-scores)[:n_recs]
    return [movies_df.iloc[i] for i in top]

# small example
watched = [0, 1]   # change to indexes you know
print("Watched:", movies_df.iloc[watched]["Title"].tolist())
HTML(movie_display.show(recommend_from_watched(watched, 5)))

Watched: ['Toy Story', 'Jumanji']
